In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
# ════════════════════════════════════════════════════════════════════════════════
#  India PM2.5 Forecasting — v9
#  Base: 0.8815 notebook
#  Changes:
#    1. Stacked 2-layer ConvLSTM (32→64 hidden) — deeper temporal encoding
#    2. ConvLSTM also runs on met features (wind, pblh) — not just PM2.5
#    3. UNet base widened 64→96 — more spatial capacity
#  Everything else IDENTICAL: 170ch, EMA, 4-flip TTA, dynamic epw, HV aug
# ════════════════════════════════════════════════════════════════════════════════
import os, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from pathlib import Path
from copy import deepcopy
from datetime import datetime
from tqdm import tqdm

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# ── Seeds ─────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
os.environ['PYTHONHASHSEED'] = str(SEED)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f"Device: {DEVICE} | GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = Path('/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2')
RAW  = BASE / 'raw'
TEST = BASE / 'test_in'

MONTH_DIRS = {
    'April'   : RAW / 'APRIL_16',
    'July'    : RAW / 'JULY_16',
    'October' : RAW / 'OCT_16',
    'December': RAW / 'DEC_16',
}

# ── Features (identical to 0.8815) ────────────────────────────────────────────
MET_FEATURES = ['u10', 'v10', 't2', 'q2', 'psfc', 'pblh', 'rain', 'swdown']
EMI_FEATURES = ['PM25', 'NH3', 'SO2', 'NOx', 'NMVOCe', 'bio']
ALL_FEATURES = ['cpm25'] + MET_FEATURES + EMI_FEATURES

T_IN  = 10
T_MET = 10
T_OUT = 16

# ConvLSTM runs on PM2.5 (1ch) + wind+pblh (3ch) = 4ch input
# 2-layer stacked: hidden 32 → 64, output = 64
LSTM_IN_CH  = 4      # cpm25 + u10 + v10 + pblh (most physically relevant)
LSTM_H1     = 32     # layer 1 hidden
LSTM_H2     = 64     # layer 2 hidden (output)

IN_CH      = T_IN + T_MET*len(MET_FEATURES) + T_MET*len(EMI_FEATURES) + T_MET*2  # 170
UNET_IN_CH = IN_CH + 1 + LSTM_H2   # 170 + 1 trend + 64 LSTM = 235
print(f"Raw input channels: {IN_CH}  |  UNet input: {UNET_IN_CH}")

# ── Config ────────────────────────────────────────────────────────────────────
CFG = dict(
    epochs        = 80,
    batch_size    = 16,
    lr            = 5e-4,
    weight_decay  = 1e-4,
    grad_clip     = 1.0,
    patience      = 15,
    warmup_epochs = 5,
    ema_decay     = 0.999,
)

# ── Normalisation (identical) ─────────────────────────────────────────────────
LOG_FEATS = {'cpm25', 'rain', 'pblh', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOCe', 'bio'}

def compute_norm_stats():
    rng = np.random.default_rng(SEED); stats = {}
    print("Computing norm stats...")
    for feat in ALL_FEATURES:
        samples = []; use_log = feat in LOG_FEATS
        for d in MONTH_DIRS.values():
            path = d / f'{feat}.npy'
            if not path.exists(): continue
            arr = np.load(path, mmap_mode='r').ravel().astype(np.float32)
            if use_log: arr = np.log1p(arr.clip(0))
            idx = rng.choice(len(arr), max(1, min(int(len(arr)*0.05), 500000)), replace=False)
            samples.append(arr[idx])
        if not samples:
            stats[feat] = {'log': use_log, 'mean': 0.0, 'std': 1.0}
            print(f'  {feat:<14} MISSING — identity stats'); continue
        vals = np.concatenate(samples)
        stats[feat] = {'log': use_log, 'mean': float(vals.mean()),
                       'std': float(vals.std().clip(1e-8))}
        print(f'  {feat:<14} mean={stats[feat]["mean"]:>10.4f}  '
              f'std={stats[feat]["std"]:>9.4f}  log={use_log}')
    return stats

def normalize(arr, s):
    if s['log']: arr = np.log1p(arr.clip(0))
    return (arr - s['mean']) / s['std']

def denormalize(arr, s):
    out = arr * s['std'] + s['mean']
    if s['log']: out = np.expm1(np.clip(out, -9., 9.)).clip(0)
    return out

# ── STL Episode Masks (identical) ─────────────────────────────────────────────
def compute_episode_mask(month_dir):
    cpm25 = np.load(Path(month_dir) / 'cpm25.npy').astype(np.float32)
    T, H, W = cpm25.shape
    trend = np.empty_like(cpm25); pad = 12
    for hi in range(H):
        for wi in range(W):
            s = np.pad(cpm25[:, hi, wi], pad, mode='edge')
            trend[:, hi, wi] = np.convolve(s, np.ones(25)/25, 'valid')
    detrended = cpm25 - trend
    diurnal   = np.zeros((24, H, W), dtype=np.float32)
    counts    = np.zeros(24, dtype=np.int32)
    for t in range(T):
        h = t % 24; diurnal[h] += detrended[t]; counts[h] += 1
    for h in range(24):
        if counts[h]: diurnal[h] /= counts[h]
    diurnal_full = np.stack([diurnal[t % 24] for t in range(T)])
    residual     = detrended - diurnal_full
    sigma        = residual.std(axis=0).clip(1e-8)
    mask         = residual > 3.0 * sigma[np.newaxis]
    print(f'episode={mask.mean()*100:.3f}%  sigma_mean={sigma.mean():.2f}  sigma_max={sigma.max():.2f}')
    return mask

# ── Metrics (identical) ───────────────────────────────────────────────────────
def smape_np(p, y, eps=1.0):
    return float(np.mean(2*np.abs(p-y) / (np.abs(p)+np.abs(y)+eps)))

def phase2_metrics(pred_r, yr, ep_mask_np):
    B, T, H, W = pred_r.shape
    gsmapes, epcorrs, epsmapes = [], [], []
    for t in range(T):
        gsmapes.append(smape_np(pred_r[:,t].ravel(), yr[:,t].ravel()))
        ept = ep_mask_np[:, t]
        if ept.any():
            pep = pred_r[:,t][ept]; yep = yr[:,t][ept]
            if len(pep) > 1:
                c = float(np.corrcoef(pep, yep)[0,1])
                if not np.isnan(c):
                    epcorrs.append(c); epsmapes.append(smape_np(pep, yep))
    gs = float(np.mean(gsmapes))
    ec = float(np.mean(epcorrs)) if epcorrs else 0.0
    es = float(np.mean(epsmapes)) if epsmapes else gs
    return {'gsmape': gs, 'ecorr': ec, 'esmape': es,
            'score': ((1-gs/2) + (ec+1)/2 + (1-es/2)) / 3}

# ── Loss — dynamic epw 0.20→0.35 (identical) ─────────────────────────────────
def smape_fn(pred, target, eps=1.0):
    return (2*(pred-target).abs() / (pred.abs()+target.abs()+eps)).mean()

def composite_loss(pred, target, ep_mask, epoch=0, total_epochs=80):
    progress = min(epoch / max(total_epochs, 1), 1.0)
    epw      = 0.20 + 0.15*progress
    huberw   = 0.50 - 0.05*progress
    smapew   = 1.0 - huberw - epw
    huber    = F.huber_loss(pred, target, delta=1.0)
    smape    = smape_fn(pred, target)
    ep_bool  = ep_mask.bool()
    ep_loss  = (F.huber_loss(pred[ep_bool], target[ep_bool], delta=0.5) +
                smape_fn(pred[ep_bool], target[ep_bool])) if ep_bool.any() else huber
    return huberw*huber + smapew*smape + epw*ep_loss

# ── Dataset (identical — HV aug, STL masks) ───────────────────────────────────
class PM25Dataset(Dataset):
    def __init__(self, month_dir, stride=2, t_start=0, t_end=None,
                 episode_mask=None, augment=False):
        month_dir = Path(month_dir)
        ref_shape = np.load(month_dir / 'cpm25.npy', mmap_mode='r').shape
        self.mmap = {}
        for f in ALL_FEATURES:
            p = month_dir / f'{f}.npy'
            self.mmap[f] = np.load(p, mmap_mode='r') if p.exists() else \
                           np.zeros(ref_shape, dtype=np.float32)
        time_arr = np.load(month_dir / 'time.npy', allow_pickle=True)
        try:
            self.hours = np.array([datetime.strptime(str(s), '%Y-%m-%d_%H:%M:%S').hour
                                   for s in time_arr], dtype=np.float32)
        except:
            self.hours = (np.arange(len(time_arr)) % 24).astype(np.float32)
        T, H, W = ref_shape; t_end = t_end or T
        self.H = H; self.W = W; self.augment = augment; self.ep_mask = episode_mask
        self.indices = [i for i in range(0, T-T_IN-T_OUT+1, stride)
                        if t_start <= i and i+T_IN+T_OUT <= t_end]

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        t0 = self.indices[idx]; H, W = self.H, self.W
        fliph = self.augment and random.random() < 0.5
        flipv = self.augment and random.random() < 0.7

        def load(feat, start, length):
            a = self.mmap[feat][start:start+length].astype(np.float32)
            a = normalize(a, norm_stats[feat])
            if fliph: a = a[..., ::-1].copy()
            if flipv: a = a[..., ::-1, :].copy()
            return a

        pm     = load('cpm25', t0, T_IN)
        metemi = np.concatenate([load(f, t0, T_MET) for f in MET_FEATURES+EMI_FEATURES], axis=0)
        hw     = self.hours[t0:t0+T_MET]
        sin_h  = np.sin(2*np.pi*hw/24).reshape(-1,1,1) * np.ones((1,H,W), np.float32)
        cos_h  = np.cos(2*np.pi*hw/24).reshape(-1,1,1) * np.ones((1,H,W), np.float32)
        x      = np.concatenate([pm, metemi, sin_h, cos_h], axis=0)

        yraw = self.mmap['cpm25'][t0+T_IN:t0+T_IN+T_OUT].astype(np.float32)
        if fliph: yraw = yraw[..., ::-1].copy()
        if flipv: yraw = yraw[..., ::-1, :].copy()
        y = normalize(yraw, norm_stats['cpm25'])

        ep = (self.ep_mask[t0+T_IN:t0+T_IN+T_OUT].astype(np.float32)
              if self.ep_mask is not None else np.zeros((T_OUT,H,W), np.float32))
        if self.ep_mask is not None and fliph: ep = ep[..., ::-1].copy()
        if self.ep_mask is not None and flipv: ep = ep[..., ::-1, :].copy()

        return (torch.from_numpy(x), torch.from_numpy(y),
                torch.from_numpy(pm[-1]), torch.from_numpy(ep))


class TestDataset(Dataset):
    def __init__(self):
        ref = np.load(TEST / 'cpm25.npy', mmap_mode='r')
        self.N, _, self.H, self.W = ref.shape
        self.mmap = {}
        for f in ALL_FEATURES:
            p = TEST / f'{f}.npy'
            self.mmap[f] = np.load(p, mmap_mode='r') if p.exists() else \
                           np.zeros((self.N, T_MET, self.H, self.W), np.float32)
        print(f'TestDataset: N={self.N}, H={self.H}, W={self.W}')

    def __len__(self): return self.N

    def __getitem__(self, idx):
        H, W = self.H, self.W
        pm     = normalize(self.mmap['cpm25'][idx].astype(np.float32), norm_stats['cpm25'])
        metemi = np.concatenate([normalize(self.mmap[f][idx].astype(np.float32), norm_stats[f])
                                 for f in MET_FEATURES+EMI_FEATURES], axis=0)
        hw    = ((idx % 24) + np.arange(T_MET)) % 24
        sin_h = np.sin(2*np.pi*hw/24).astype(np.float32).reshape(-1,1,1)*np.ones((1,H,W),np.float32)
        cos_h = np.cos(2*np.pi*hw/24).astype(np.float32).reshape(-1,1,1)*np.ones((1,H,W),np.float32)
        return torch.from_numpy(np.concatenate([pm, metemi, sin_h, cos_h], axis=0)), \
               torch.from_numpy(pm[-1])

# ════════════════════════════════════════════════════════════════════════════════
#  MODEL — v9 changes:
#   1. 2-layer stacked ConvLSTM (32→64) on PM2.5 + wind + pblh
#   2. UNet base widened 64→96
#   Everything else identical to 0.8815
# ════════════════════════════════════════════════════════════════════════════════

class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size=3):
        super().__init__()
        self.hidden_ch = hidden_ch
        pad = kernel_size // 2
        self.conv = nn.Conv2d(in_ch + hidden_ch, 4*hidden_ch,
                              kernel_size, padding=pad, bias=True)
        self.norm = nn.GroupNorm(min(8, hidden_ch), hidden_ch)

    def forward(self, x, h, c):
        gates      = self.conv(torch.cat([x, h], dim=1))
        i, f, g, o = gates.chunk(4, dim=1)
        c_new = torch.sigmoid(f)*c + torch.sigmoid(i)*torch.tanh(g)
        h_new = torch.sigmoid(o) * torch.tanh(self.norm(c_new))
        return h_new, c_new

    def init_hidden(self, B, H, W, device):
        return (torch.zeros(B, self.hidden_ch, H, W, device=device),
                torch.zeros(B, self.hidden_ch, H, W, device=device))


class StackedConvLSTM(nn.Module):
    """
    2-layer stacked ConvLSTM.
    Input:  (B, T, C_in, H, W)  — C_in = 4 (pm25 + u10 + v10 + pblh)
    Output: (B, LSTM_H2, H, W)  — final hidden state of layer 2
    """
    def __init__(self, in_ch=LSTM_IN_CH, h1=LSTM_H1, h2=LSTM_H2):
        super().__init__()
        self.cell1 = ConvLSTMCell(in_ch, h1, kernel_size=3)
        self.cell2 = ConvLSTMCell(h1,    h2, kernel_size=3)
        self.proj  = nn.Sequential(
            nn.Conv2d(h2, h2, 1),
            nn.GroupNorm(min(8, h2), h2),
            nn.GELU())

    def forward(self, seq):
        B, T, C, H, W = seq.shape
        h1, c1 = self.cell1.init_hidden(B, H, W, seq.device)
        h2, c2 = self.cell2.init_hidden(B, H, W, seq.device)
        for t in range(T):
            h1, c1 = self.cell1(seq[:, t], h1, c1)
            h2, c2 = self.cell2(h1, h2, c2)
        return self.proj(h2)   # (B, LSTM_H2, H, W)

# ── Spatial components (identical to 0.8815) ──────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ch, max(1,ch//r)), nn.ReLU(),
                                nn.Linear(max(1,ch//r), ch))
    def forward(self, x):
        w = torch.sigmoid(self.fc(x.mean([2,3])) + self.fc(x.amax([2,3])))
        return x * w.unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        return x * torch.sigmoid(self.conv(
            torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], 1)))

class CBAM(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.ca = ChannelAttention(ch); self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))

class SpatialSelfAttention(nn.Module):
    def __init__(self, ch, n_heads=8):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = ch // n_heads
        self.scale    = self.head_dim ** -0.5
        self.qkv      = nn.Conv2d(ch, ch*3, 1, bias=False)
        self.proj     = nn.Conv2d(ch, ch, 1)
        self.norm     = nn.GroupNorm(min(8,ch), ch)

    def forward(self, x):
        B, C, H, W = x.shape; N = H*W
        qkv = self.qkv(x).reshape(B, 3, self.n_heads, self.head_dim, N)
        q, k, v = qkv.unbind(1)
        attn = torch.softmax(torch.einsum('bhdn,bhdm->bhnm', q, k)*self.scale, dim=-1)
        out  = torch.einsum('bhnm,bhdm->bhdn', attn, v).reshape(B, C, H, W)
        return x + self.proj(self.norm(out))

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8,out_ch), out_ch), nn.GELU(),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8,out_ch), out_ch), nn.GELU())
    def forward(self, x): return self.net(x)

# ── EMA (identical) ───────────────────────────────────────────────────────────
class EMA:
    def __init__(self, model, decay=0.999):
        m = model.module if isinstance(model, nn.DataParallel) else model
        self.model = deepcopy(m).eval(); self.decay = decay

    @torch.no_grad()
    def update(self, model):
        m = model.module if isinstance(model, nn.DataParallel) else model
        for ep, p in zip(self.model.parameters(), m.parameters()):
            ep.data.mul_(self.decay).add_(p.data, alpha=1-self.decay)

    def eval(self): self.model.eval(); return self.model

# ── ImprovedSkipUNet v9 ───────────────────────────────────────────────────────
class ImprovedSkipUNet(nn.Module):
    def __init__(self, in_ch=IN_CH, base=96, dropout=0.10):   # base 64→96
        super().__init__()
        b, DR = base, dropout

        # ── 2-layer stacked ConvLSTM on pm25 + u10 + v10 + pblh ────────────
        self.temporal_enc = StackedConvLSTM(in_ch=LSTM_IN_CH, h1=LSTM_H1, h2=LSTM_H2)
        unet_in = in_ch + 1 + LSTM_H2   # 170 + trend + 64 = 235

        # ── UNet (base=96, identical structure) ──────────────────────────────
        self.enc1       = DoubleConv(unet_in, b,   DR)
        self.enc2       = DoubleConv(b,   b*2, DR)
        self.enc3       = DoubleConv(b*2, b*4, DR)
        self.bottleneck = nn.Sequential(
            DoubleConv(b*4, b*8, DR),
            CBAM(b*8),
            SpatialSelfAttention(b*8, n_heads=8))
        self.pool = nn.MaxPool2d(2)
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = nn.Sequential(DoubleConv(b*8, b*4, DR), CBAM(b*4))
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = nn.Sequential(DoubleConv(b*4, b*2, DR), CBAM(b*2))
        self.up1  = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.dec1 = DoubleConv(b*2, b)
        self.out  = nn.Conv2d(b, T_OUT, 1)

    def forward(self, x, last_pm25):
        # Trend delta (identical to 0.8815)
        trend = x[:, T_IN-1:T_IN] - x[:, T_IN-3:T_IN-2]

        # Build ConvLSTM input: pm25 + u10 + v10 + pblh — each (B,T,H,W)
        # In x layout: [0:T_IN]=pm25, [T_IN:T_IN+T_MET]=u10, [T_IN+T_MET:T_IN+2*T_MET]=v10
        # pblh is met feature index 5 → starts at T_IN + 5*T_MET
        pm_seq   = x[:, :T_IN].unsqueeze(2)                           # (B,T,1,H,W)
        u10_seq  = x[:, T_IN:T_IN+T_MET].unsqueeze(2)                 # (B,T,1,H,W)
        v10_seq  = x[:, T_IN+T_MET:T_IN+2*T_MET].unsqueeze(2)        # (B,T,1,H,W)
        pblh_seq = x[:, T_IN+5*T_MET:T_IN+6*T_MET].unsqueeze(2)      # (B,T,1,H,W)
        lstm_in  = torch.cat([pm_seq, u10_seq, v10_seq, pblh_seq], dim=2)  # (B,T,4,H,W)

        ctx   = self.temporal_enc(lstm_in)          # (B, 64, H, W)
        x_in  = torch.cat([x, trend, ctx], dim=1)   # (B, 235, H, W)

        e1 = self.enc1(x_in)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bn = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([F.interpolate(self.up3(bn), e3.shape[2:]), e3], 1))
        d2 = self.dec2(torch.cat([F.interpolate(self.up2(d3), e2.shape[2:]), e2], 1))
        d1 = self.dec1(torch.cat([F.interpolate(self.up1(d2), e1.shape[2:]), e1], 1))
        return self.out(d1) + last_pm25.unsqueeze(1)

# ── Scheduler (identical) ─────────────────────────────────────────────────────
def get_scheduler(opt, warmup_eps, total_eps):
    def lr_fn(ep):
        if ep < warmup_eps: return (ep+1) / warmup_eps
        p = (ep-warmup_eps) / max(1, total_eps-warmup_eps)
        return 0.5*(1+np.cos(np.pi*p))*(1-0.02) + 0.02
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

# ── Train (identical) ─────────────────────────────────────────────────────────
def train_model(model, name, train_ldr, val_ldr, cfg):
    if NUM_GPUS > 1: model = nn.DataParallel(model); print(f'DataParallel on {NUM_GPUS} GPUs')
    model.to(DEVICE)
    ema    = EMA(model, decay=cfg['ema_decay'])
    opt    = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sch    = get_scheduler(opt, cfg['warmup_epochs'], cfg['epochs'])
    scaler = torch.amp.GradScaler('cuda')
    s      = norm_stats['cpm25']
    best_score = -float('inf'); pat = 0
    best_path  = f'/kaggle/working/{name}_best.pt'

    hdr = f"{'Ep':>4} {'TrL':>7} {'VaL':>7} {'gSMAPE':>8} {'eCorr':>7} {'eSMAPE':>8} {'Score':>7} {'LR':>8}"
    print(f'\n{hdr}'); print('-'*len(hdr))

    for epoch in range(cfg['epochs']):
        t0 = time.time(); model.train(); tr_loss = 0.0
        for x, y, last, ep in tqdm(train_ldr, desc=f'Ep{epoch+1:02d}', leave=False):
            x, y, last, ep = x.to(DEVICE), y.to(DEVICE), last.to(DEVICE), ep.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = composite_loss(model(x, last), y, ep,
                                      epoch=epoch, total_epochs=cfg['epochs'])
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            m = model.module if isinstance(model, nn.DataParallel) else model
            nn.utils.clip_grad_norm_(m.parameters(), cfg['grad_clip'])
            scaler.step(opt); scaler.update()
            ema.update(model); tr_loss += loss.item()
        sch.step()

        ema_model = ema.eval()
        va_loss = 0.0; p_list, y_list, ep_list = [], [], []
        with torch.no_grad():
            for x, y, last, ep in val_ldr:
                x, y, last, ep = x.to(DEVICE), y.to(DEVICE), last.to(DEVICE), ep.to(DEVICE)
                with torch.amp.autocast('cuda'):
                    pred    = ema_model(x, last)
                    va_loss += composite_loss(pred, y, ep, epoch=epoch,
                                             total_epochs=cfg['epochs']).item()
                p_list.append(pred.float().cpu().numpy())
                y_list.append(y.float().cpu().numpy())
                ep_list.append(ep.float().cpu().numpy())

        p_all   = np.concatenate(p_list); y_all = np.concatenate(y_list)
        ep_all  = np.concatenate(ep_list).astype(bool)
        metrics = phase2_metrics(denormalize(p_all, s), denormalize(y_all, s), ep_all)
        score   = metrics['score']; cur_lr = opt.param_groups[0]['lr']

        is_best = score > best_score
        if is_best:
            best_score = score; pat = 0
            core = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(core.state_dict(), best_path)
        else:
            pat += 1

        flag = ' ✓' if is_best else ''
        print(f"{epoch+1:>4} {tr_loss/len(train_ldr):>7.4f} {va_loss/len(val_ldr):>7.4f} "
              f"{metrics['gsmape']:>8.4f} {metrics['ecorr']:>7.4f} "
              f"{metrics['esmape']:>8.4f} {score:>7.4f} {cur_lr:>8.2e}  "
              f"{(time.time()-t0)/60:.1f}m{flag}")

        if epoch+1 in [10, 20, 30, 40, 50, 60, 70]:
            cp = Path(f'/kaggle/working/{name}_ep{epoch+1:03d}.pt')
            core = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(core.state_dict(), cp)
            print(f'  ckpt {cp.name}')

        if pat >= cfg['patience']:
            print(f'Early stop ep{epoch+1}. Best={best_score:.4f}'); break

    core = model.module if isinstance(model, nn.DataParallel) else model
    core.load_state_dict(torch.load(best_path, map_location='cpu'))
    print(f'\nBest EMA score={best_score:.4f}')
    return core, best_score

# ── TTA — 4-flip batched (identical) ─────────────────────────────────────────
AUG_CONFIGS = [(fh, fv) for fh in [False, True] for fv in [False, True]]

def tta_batch(xnp, lastnp):
    B = xnp.shape[0]; aug_x, aug_l = [], []
    for fh, fv in AUG_CONFIGS:
        xf = xnp.copy(); lf = lastnp.copy()
        if fh: xf = xf[..., ::-1].copy();  lf = lf[..., ::-1].copy()
        if fv: xf = xf[..., ::-1, :].copy(); lf = lf[..., ::-1, :].copy()
        aug_x.append(xf); aug_l.append(lf)
    X = torch.from_numpy(np.concatenate(aug_x)).to(DEVICE)
    L = torch.from_numpy(np.concatenate(aug_l)).to(DEVICE)
    with torch.no_grad(), torch.amp.autocast('cuda'):
        pred_norm = ema_model(X, L).float().cpu().numpy()
    preds = []
    for i, (fh, fv) in enumerate(AUG_CONFIGS):
        p = pred_norm[i*B:(i+1)*B].copy()
        if fh: p = p[..., ::-1]
        if fv: p = p[..., ::-1, :]
        preds.append(p)
    return denormalize(np.mean(preds, axis=0), norm_stats['cpm25'])

# ════════════════════════════════════════════════════════════════════════════════
#                               MAIN
# ════════════════════════════════════════════════════════════════════════════════
norm_stats = compute_norm_stats()

print('\nComputing STL episode masks (official definition ri,t > 3*sigma_i)...')
ep_masks = {}
for mname, mdir in MONTH_DIRS.items():
    print(f'  {mname}: ', end='', flush=True)
    ep_masks[mname] = compute_episode_mask(mdir)

OCT_T     = np.load(MONTH_DIRS['October'] / 'cpm25.npy', mmap_mode='r').shape[0]
val_start = max(0, OCT_T - 200)
print(f'\nOCT={OCT_T}h  Val=[{val_start},{OCT_T}]={OCT_T-val_start}h')

train_ds = ConcatDataset([
    PM25Dataset(MONTH_DIRS['April'],    stride=2, augment=True,  episode_mask=ep_masks['April']),
    PM25Dataset(MONTH_DIRS['July'],     stride=2, augment=True,  episode_mask=ep_masks['July']),
    PM25Dataset(MONTH_DIRS['December'], stride=2, augment=True,  episode_mask=ep_masks['December']),
    PM25Dataset(MONTH_DIRS['October'],  stride=2, t_end=val_start,
                augment=True, episode_mask=ep_masks['October']),
])
val_ds  = PM25Dataset(MONTH_DIRS['October'], stride=2, t_start=val_start,
                      episode_mask=ep_masks['October'])
test_ds = TestDataset()
print(f'Train={len(train_ds)}  Val={len(val_ds)}  Test={len(test_ds)}')

def seed_worker(wid): np.random.seed(SEED+wid); random.seed(SEED+wid)
g = torch.Generator(); g.manual_seed(SEED)

train_ldr = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True,
                       worker_init_fn=seed_worker, generator=g)
val_ldr   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                       num_workers=2, pin_memory=True)
test_ldr  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                       num_workers=2, pin_memory=True)

xb, yb, lb, eb = next(iter(train_ldr))
print(f'\nBatch: x={xb.shape} y={yb.shape} last={lb.shape} ep={eb.shape}')
assert xb.shape[1] == IN_CH, f"Expected IN_CH={IN_CH}, got {xb.shape[1]}"

# Verify forward pass
model = ImprovedSkipUNet(IN_CH, base=96, dropout=0.10)
with torch.no_grad():
    out = model(xb, lb)
assert out.shape == (xb.shape[0], T_OUT, xb.shape[2], xb.shape[3])
total_p = sum(p.numel() for p in model.parameters())
lstm_p  = sum(p.numel() for p in model.temporal_enc.parameters())
print(f'Model fwd OK ✓  |  Total params: {total_p:,}  (StackedConvLSTM: {lstm_p:,})')
del out

model, best_score = train_model(model, 'v9', train_ldr, val_ldr, CFG)

# ── TTA Inference (identical) ─────────────────────────────────────────────────
print('\nRunning 4-flip batched TTA inference...')
ema_model = model; ema_model.eval()
all_preds = []; INFER_BS = 16

for start in tqdm(range(0, len(test_ds), INFER_BS), desc='TTA Inference'):
    end = min(start+INFER_BS, len(test_ds))
    xs, ls = [], []
    for i in range(start, end):
        xi, li = test_ds[i]; xs.append(xi.numpy()); ls.append(li.numpy())
    all_preds.append(tta_batch(np.stack(xs), np.stack(ls)))

preds     = np.concatenate(all_preds).clip(0, 500)
preds_out = preds.transpose(0, 2, 3, 1)   # (218, 140, 124, 16)
assert preds_out.shape[1:] == (140, 124, T_OUT), f'Shape error: {preds_out.shape}'
np.save('/kaggle/working/preds.npy', preds_out)

print(f'\nPrediction stats')
print(f'  shape={preds_out.shape}')
print(f'  mean={preds.mean():.2f}  median={np.median(preds):.2f}  '
      f'p95={np.percentile(preds,95):.1f}  max={preds.max():.1f}  NaNs={np.isnan(preds).sum()}')
print(f'\nSaved /kaggle/working/preds.npy')
print(f'Best val score: {best_score:.4f}')
print('Done ✓')

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

   1  0.1948  0.2281   0.3918  0.8090   0.5387  0.8131 2.00e-04  2.7m ✓


   2  0.1775  0.2016   0.3519  0.8330   0.4798  0.8336 3.00e-04  2.3m ✓


   3  0.1606  0.1819   0.3220  0.8539   0.4338  0.8497 4.00e-04  2.3m ✓


   4  0.1484  0.1668   0.2989  0.8689   0.3998  0.8617 5.00e-04  2.3m ✓


...

  78  0.0709  0.0961   0.1844  0.9232   0.1954  0.9239 1.09e-05  2.3m ✓


  79  0.0710  0.0962   0.1844  0.9233   0.1953  0.9239 1.02e-05  2.3m ✓


  80  0.0707  0.0963   0.1844  0.9233   0.1952  0.9240 1.00e-05  2.3m ✓

Best EMA score=0.9240

Running 4-flip batched TTA inference...


TTA Inference: 100%|██████████| 14/14 [01:12<00:00,  5.20s/it]



Prediction stats
  shape=(218, 140, 124, 16)
  mean=37.19  median=18.18  p95=132.3  max=500.0  NaNs=0

Saved /kaggle/working/preds.npy
Best val score: 0.9240
Done ✓
